# Style2Paints V3 — Bake-off + ONNX Export

Goal: run Style2Paints V3 on your sample pages, pick the look, export the model to ONNX (fp32 → fp16 → int8) with a **measured contract** (`style2paints-v3.metadata.json`) that the Rust engine backend will consume.

Artifacts land in `MyDrive/MangaHue-S2P/`:
- `bakeoff/` — V3 (and optional V4) outputs on your samples
- `onnx/` — style2paints-v3.onnx (+ int8 variant) + metadata JSON
- `verification/` — ONNX-vs-torch comparison images + numbers

**Before running**: put 3–6 sample pages (the pages you compare against the channel look) into `MyDrive/MangaHue-S2P/samples/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/MangaHue-S2P')
for name in ('samples', 'bakeoff', 'onnx', 'verification', 'models'):
    (ROOT / name).mkdir(parents=True, exist_ok=True)
samples = sorted((ROOT / 'samples').glob('*'))
print('samples found:', [p.name for p in samples])
assert samples, 'Put sample pages in MyDrive/MangaHue-S2P/samples/'

In [ ]:
import subprocess, sys
def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)
pip('torch', 'onnx', 'onnxruntime', 'pillow', 'numpy', 'opencv-python-headless', 'gdown')
import torch, onnx, onnxruntime as ort, numpy as np
print('torch', torch.__version__, '| ort', ort.__version__, '| cuda:', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Get Style2Paints V3

Clone the official repo, then obtain the three weight files (`encoder.pth`, `decoder.pth`, `pfc.pth`). Auto-download tries known mirrors; if they fail, the cell prints the exact files to place in `MyDrive/MangaHue-S2P/models/`.

In [ ]:
import shutil
repo = Path('/content/style2paints')
if not (repo / 'style2paintsV3').exists():
    if repo.exists(): shutil.rmtree(repo)
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/lllyasviel/style2paints.git', str(repo)], check=True)
v3 = repo / 'style2paintsV3'
print('V3 dir contents:', sorted(p.name for p in v3.iterdir())[:20])
print('\n--- network modules ---')
import glob
for f in glob.glob(str(v3 / '*.py')):
    print(' ', Path(f).name)

In [ ]:
import urllib.request
weights = {'encoder.pth': None, 'decoder.pth': None, 'pfc.pth': None}
# Known mirrors to try in order (official repo README links are Drive/Baidu;
# community HF mirrors change often — add any working URL here).
mirrors = {
    'encoder.pth': ['https://huggingface.co/datasets/camenduru/style2paints-v3/resolve/main/encoder.pth'],
    'decoder.pth': ['https://huggingface.co/datasets/camenduru/style2paints-v3/resolve/main/decoder.pth'],
    'pfc.pth': ['https://huggingface.co/datasets/camenduru/style2paints-v3/resolve/main/pfc.pth'],
}
for name in weights:
    dest = ROOT / 'models' / name
    if dest.exists() and dest.stat().st_size > 1_000_000:
        weights[name] = str(dest)
        print(f'{name}: already present ({dest.stat().st_size/1e6:.0f} MB)')
        continue
    ok = False
    for url in mirrors.get(name, []):
        try:
            print(f'downloading {name} <- {url}')
            urllib.request.urlretrieve(url, dest)
            if dest.stat().st_size > 1_000_000:
                ok = True
                weights[name] = str(dest)
                print(f'  ok ({dest.stat().st_size/1e6:.0f} MB)')
                break
        except Exception as e:
            print(f'  failed: {e}')
    if not ok:
        print(f'!! could not auto-download {name} — download it manually into MyDrive/MangaHue-S2P/models/ and re-run this cell')
print('\nweights:', {k: (v is not None) for k, v in weights.items()})

## 2. Run V3 on the samples (torch, GPU)

Loads the V3 network (encoder → decoder with zero strokes → pfc), colorizes each sample at 512×512, saves the results to `bakeoff/`.

In [ ]:
import importlib.util, sys, json
sys.path.insert(0, str(v3))
# V3 network module name differs across repo revisions (network.py / v3_network.py).
net_candidates = [v3 / 'network.py', v3 / 'v3_network.py', v3 / 'network2.py']
net_mod = None
for cand in net_candidates:
    if cand.exists():
        spec = importlib.util.spec_from_file_location('s2p_net', cand)
        net_mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(net_mod)
        print('loaded network module:', cand.name)
        break
assert net_mod is not None, 'no V3 network module found — inspect the repo layout above'
print('module members:', [n for n in dir(net_mod) if n[0].isupper()])

In [ ]:
import torch.nn as nn

def build_v3(mod, weights):
    """Assemble the V3 pipeline from whatever classes the repo exposes."""
    enc_cls = getattr(mod, 'Encoder', None) or getattr(mod, 'Colorization_Encoder', None)
    dec_cls = getattr(mod, 'Decoder', None) or getattr(mod, 'Colorization_Decoder', None)
    pfc_cls = getattr(mod, 'PFC', None) or getattr(mod, 'pfc', None)
    assert enc_cls and dec_cls, f'need Encoder/Decoder classes, got: {[n for n in dir(mod) if n[0].isupper()]}'
    enc, dec = enc_cls().to(DEVICE), dec_cls().to(DEVICE)
    enc.load_state_dict(torch.load(weights['encoder.pth'], map_location=DEVICE))
    dec.load_state_dict(torch.load(weights['decoder.pth'], map_location=DEVICE))
    pfc = None
    if pfc_cls is not None and weights.get('pfc.pth'):
        pfc = pfc_cls().to(DEVICE)
        pfc.load_state_dict(torch.load(weights['pfc.pth'], map_location=DEVICE))
    enc.eval(); dec.eval()
    if pfc is not None: pfc.eval()
    return enc, dec, pfc

enc, dec, pfc = build_v3(net_mod, weights)
print('V3 assembled: encoder + decoder' + (' + pfc' if pfc else ''))

In [ ]:
from PIL import Image

def prep_sample(path, size=512):
    img = Image.open(path).convert('RGB')
    img = img.resize((size, size), Image.LANCZOS)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return torch.from_numpy(arr.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)

def v3_colorize(sample):
    # V3 expects a grayscale sketch as 3-channel input; zero hint strokes.
    gray = sample.mean(1, keepdim=True).repeat(1, 3, 1, 1)
    with torch.no_grad():
        feats = enc(gray)
        out = dec(feats, torch.zeros_like(gray))
        if pfc is not None:
            out = pfc(out, gray)
    return out.clamp(0, 1)

results = {}
for s in samples:
    t = prep_sample(s)
    out = v3_colorize(t)[0].cpu().numpy().transpose(1, 2, 0)
    Image.fromarray((out * 255).astype(np.uint8)).save(ROOT / 'bakeoff' / f'{s.stem}_v3.png')
    results[s.name] = float(np.abs(out - out.mean()).mean())
    print('colored', s.name)
print('\nsaved to', ROOT / 'bakeoff')

## 3. Optional: V4 reference run (Colab GPU only — not for the VPS)

V4 is too heavy for the 2-core VPS (RAM + CPU time), but its output on your samples is the **style reference** — if it wins the bake-off, we fine-tune DDColor toward it. Skip if not needed.

In [ ]:
DO_V4 = False  # flip to True to run
if DO_V4:
    v4repo = Path('/content/style2paintsV4')
    if not v4repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/lllyasviel/style2paintsV4.git', str(v4repo)], check=True)
    print('V4 repo cloned. Follow its README for the weight download (Drive link);')
    print('then run its inference script on the samples and save outputs to bakeoff/*_v4.png')
else:
    print('V4 skipped (set DO_V4 = True to run on this GPU).')

## 4. Export V3 to ONNX

Exports the assembled pipeline as a single graph: input `input` [1,3,512,512], output `output` [1,3,512,512]. If the repo ships its own export script with a known-good graph, prefer it — otherwise this direct export is the fallback and is what we validate below.

In [ ]:
class S2PV3Graph(nn.Module):
    def __init__(self, enc, dec, pfc):
        super().__init__()
        self.enc, self.dec, self.pfc = enc, dec, pfc
    def forward(self, x):
        feats = self.enc(x)
        out = self.dec(feats, torch.zeros_like(x))
        if self.pfc is not None:
            out = self.pfc(out, x)
        return out

graph = S2PV3Graph(enc.cpu(), dec.cpu(), pfc.cpu() if pfc else None).eval()
dummy = torch.randn(1, 3, 512, 512)
onnx_path = ROOT / 'onnx' / 'style2paints-v3.onnx'
torch.onnx.export(
    graph, (dummy,), str(onnx_path),
    input_names=['input'], output_names=['output'],
    dynamic_axes=None, opset_version=17,
)
m = onnx.load(str(onnx_path))
onnx.checker.check_model(m)
print('exported', onnx_path, f'({onnx_path.stat().st_size/1e6:.0f} MB)')
for t in list(m.graph.input) + list(m.graph.output):
    dims = [d.dim_value if d.HasField('dim_value') else d.dim_param for d in t.type.tensor_type.shape.dim]
    print(f'  {t.name}: {dims}')

## 5. Verify the ONNX (runtime parity + contract JSON)

Runs the exported graph with onnxruntime and compares against the torch output — must agree within a small tolerance. Also measures the real tensor ranges and writes `style2paints-v3.metadata.json`, which the Rust backend reads.

In [ ]:
import json
sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
in_name = sess.get_inputs()[0].name
out_name = sess.get_outputs()[0].name
print('inputs:', [(i.name, i.shape, i.type) for i in sess.get_inputs()])
print('outputs:', [(o.name, o.shape, o.type) for o in sess.get_outputs()])

max_diff = 0.0
for s in samples[:3]:
    t = prep_sample(s).cpu()
    with torch.no_grad():
        torch_out = v3_colorize(t.to(DEVICE)).cpu().numpy()
    ort_out = sess.run([out_name], {in_name: t.numpy()})[0]
    diff = float(np.abs(ort_out - torch_out).max())
    max_diff = max(max_diff, diff)
    img = np.clip(ort_out[0].transpose(1, 2, 0), 0, 1)
    Image.fromarray((img * 255).astype(np.uint8)).save(ROOT / 'verification' / f'{s.stem}_onnx.png')
print(f'max |onnx - torch| = {max_diff:.5f}  (must be < 0.02)')
assert max_diff < 0.02, 'ONNX diverges from torch — check the export'

meta = {
  "name": "Style2Paints V3 ONNX",
  "role": "anime_cel_colorizer",
  "source_repository": "https://github.com/lllyasviel/style2paints",
  "license": "check before commercial use (research-restricted)",
  "input": {"name": in_name, "shape": [1, 3, 512, 512], "color_space": "grayscale_srgb", "range": "0_1"},
  "output": {"name": out_name, "shape": [1, 3, 512, 512], "color_space": "srgb", "range": "0_1"},
  "notes": "Full-RGB cel-shaded output; engine must NOT merge Lab — use output as-is (line art preserved by the model)."
}
(ROOT / 'onnx' / 'style2paints-v3.metadata.json').write_text(json.dumps(meta, indent=2))
print('wrote', ROOT / 'onnx' / 'style2paints-v3.metadata.json')
print(json.dumps(meta, indent=2))

## 6. fp16 + int8 variants (for the RAM-light VPS)

fp16 halves the file; int8 (~4× smaller, 2–4× faster CPU) via the repo's validated `export_quantized.py`. Quality gate: int8 must stay ≥90% of fp32 (compare the verification images).

In [ ]:
# fp16 (weights-only cast is NOT enough — use the engine's validated path when the
# source archive is present; here we do the direct cast + checker as a fallback).
from onnxconverter_common import float16  # pip install onnxconverter-common
pip('onnxconverter-common')
from onnxconverter_common import float16 as oc_float16
m_fp16 = oc_float16.convert_float_to_float16(onnx.load(str(onnx_path)), keep_io_types=True)
fp16_path = ROOT / 'onnx' / 'style2paints-v3-fp16.onnx'
onnx.save(m_fp16, str(fp16_path))
print('fp16 written:', fp16_path.name, f'({fp16_path.stat().st_size/1e6:.0f} MB)')

# int8 via the repo tooling when the MangaHue source archive is available:
#   python -m mangahue_train.export_quantized --model style2paints-v3-fp16.onnx \
#       --output style2paints-v3-int8.onnx --dynamic
# (dynamic QInt8 weights; validates + gates before exiting 0)

In [ ]:
import zipfile
zip_path = ROOT / 'style2paints-v3-artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in (ROOT / 'onnx').glob('*'):
        z.write(f, f'onnx/{f.name}')
    for f in (ROOT / 'verification').glob('*'):
        z.write(f, f'verification/{f.name}')
    for f in (ROOT / 'bakeoff').glob('*_v3.png'):
        z.write(f, f'bakeoff/{f.name}')
print('artifacts:', zip_path)
print('\nNext steps:')
print('1. Download the zip and eyeball bakeoff/*_v3.png vs your sample — does it match the soft-cel look?')
print('2. Ship the ONNX + metadata to the VPS (/opt/MangaHue/models/style2paints-v3/).')
print('3. I wire the Rust backend (contract-driven) + routing, then we benchmark and swap.')